# 03 - EDA of the Plant Dataset

## Role of This Notebook
This notebook studies `house_plants.csv` to turn a botanical catalog with free text into an interpretable base for recommendation. It does not model yet; it prepares criteria for the encoding stage in notebook 04.

## Questions It Answers
1. How clean or messy the plant dataset is.
2. How the light, use, and temperature variables are distributed.
3. Where textual ambiguities appear and require category normalization.
4. Which decisions must be fixed before ranking species.


In [ ]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve().parent
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PLANTS_PATH = RAW_DIR / 'house_plants.csv'
plants = pd.read_csv(PLANTS_PATH)
plants.head()

## 1. General Reading
First, we verify size, columns, and general statistics to understand which part of the dataset is numeric and which part depends on textual interpretation.


In [ ]:
print(f'Rows: {len(plants):,}')
print(f'Columns: {len(plants.columns)}')
display(plants.describe(include='all').transpose())

## 2. Quality Audit
Here we clean basic text spacing and review missing values. This stage does not yet solve conceptual normalization, but it prevents the same category from appearing separately only because of formatting differences.


In [ ]:
text_columns = ['latin', 'family', 'category', 'origin', 'climate', 'common', 'ideallight', 'toleratedlight', 'watering', 'use']
plants[text_columns] = plants[text_columns].apply(lambda col: col.astype(str).str.strip())

data_quality = pd.DataFrame({
    'dtype': plants.dtypes.astype(str),
    'missing_values': plants.isna().sum(),
    'n_unique': plants.nunique()
})
display(data_quality)
data_quality.to_csv(PROCESSED_DIR / 'plants_data_quality.csv')

## 3. Key Variables for Recommendation
The top 5 recommendation relies mainly on light and use. Temperature is kept, but as a weak contextual criterion, because the spatial dataset does not provide temperature per cell.


In [ ]:
for column in ['ideallight', 'toleratedlight', 'use', 'category', 'climate']:
    print(f'\n## {column}')
    display(plants[column].value_counts(dropna=False).head(15).to_frame('count'))

## 4. Repetitions and Cultivars
Not every repetition is an error. In botany, it is normal to find several rows with the same genus or base species and different cultivars. This step helps understand whether the ranking could be dominated by highly repeated families.


In [ ]:
repeated_latin = plants['latin'].value_counts().loc[lambda s: s > 1].rename_axis('latin').reset_index(name='count')
display(repeated_latin.head(20))
repeated_latin.to_csv(PROCESSED_DIR / 'plants_repeated_latin.csv', index=False)

## 5. Temperature as a Contextual Criterion
Temperature does not lead the recommendation because the current spatial problem is centered on light. Even so, it is useful to describe it to make clear that it is not ignored by oversight, but by a conscious methodological decision.


In [ ]:
temp_summary = plants[['tempmin_celsius', 'tempmax_celsius']].describe().transpose()
display(temp_summary)
temp_summary.to_csv(PROCESSED_DIR / 'plants_temperature_summary.csv')

## 6. Decisions for Notebook 04
This closing section summarizes how the next encoding stage will be approached.


In [ ]:
eda_notes = pd.Series({
    'dominant_ideal_light': plants['ideallight'].value_counts().idxmax(),
    'dominant_tolerated_light': plants['toleratedlight'].value_counts().idxmax(),
    'dominant_use': plants['use'].value_counts().idxmax(),
    'decision_1': 'The light variable will be normalized into a small number of classes to remain compatible with spatial_label.',
    'decision_2': 'Indoor use will be converted into a score because not all use texts have the same value for indoor suitability.',
    'decision_3': 'Temperature will be kept as a weak contextual criterion in the final justification.'
})
display(eda_notes.to_frame('value'))
eda_notes.to_csv(PROCESSED_DIR / 'plants_eda_notes.csv', header=['value'])